# Build knowledge graph for datasets from dynamical.org catalog


https://dynamical.org/catalog/


In [ ]:
%env AWS_PROFILE=kmnlp

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
import json
from datetime import UTC, datetime, timedelta
from textwrap import dedent
from typing import Literal

import xarray as xr
from e84_geoai_common.llm.core.llm import LLM, LLMInferenceConfig, LLMMessage
from e84_geoai_common.llm.models.claude import CLAUDE_4_5_SONNET, BedrockClaudeLLM
from pydantic import AliasChoices, BaseModel, Field, RootModel
from rich import print as rprint
from rich.pretty import pretty_repr
from tqdm import tqdm


In [ ]:
# If True, create only a single global "canonical" instance of each dim without
# the noisy long_name and units metadata fields.
# If False, create an instance for each occurence of a dim i.e. there will be
# a unique time dim for each dataset. These will also include noisy metadata
# fields.
USE_CANONICAL_DIMS = True

assert USE_CANONICAL_DIMS, "Only USE_CANONICAL_DIMS=True is supported currently."

In [ ]:
uris = {
    "forecast": [
        "https://data.dynamical.org/noaa/gfs/forecast/latest.zarr",
        "https://data.dynamical.org/noaa/gefs/forecast-35-day/latest.zarr",
        "https://data.dynamical.org/noaa/hrrr/forecast-48-hour/latest.zarr",
        "https://data.dynamical.org/ecmwf/ifs-ens/forecast-15-day-0-25-degree/latest.zarr",
    ],
    "historical": [
        "https://data.dynamical.org/noaa/gfs/analysis-hourly/latest.zarr",
        "https://data.dynamical.org/noaa/gefs/analysis/latest.zarr",
        "https://data.dynamical.org/noaa/hrrr/analysis/latest.zarr",
    ],
}

---


## Custom overrides


In [ ]:
dim_std_names = {
    "init_time": "forecast_reference_time",
    "lead_time": "forecast_period",
    "ensemble_member": "realization",
}

In [ ]:
dim_alt_names = {
    "latitude": "lat",
    "longitude": "lon",
    "init_time": "time",
}

In [ ]:
overrides = {
    "https://data.dynamical.org/noaa/gfs/forecast/latest.zarr": {
        "landing_page": "https://dynamical.org/catalog/noaa-gfs-forecast/",
    },
    "https://data.dynamical.org/noaa/gefs/forecast-35-day/latest.zarr": {
        "landing_page": "https://dynamical.org/catalog/noaa-gefs-forecast-35-day",
    },
    "https://data.dynamical.org/noaa/hrrr/forecast-48-hour/latest.zarr": {
        "landing_page": "https://dynamical.org/catalog/noaa-hrrr-forecast-48-hour",
    },
    "https://data.dynamical.org/ecmwf/ifs-ens/forecast-15-day-0-25-degree/latest.zarr": {
        "landing_page": "https://dynamical.org/catalog/ecmwf-ifs-ens-forecast-15-day-0-25-degree",
    },
    "https://data.dynamical.org/noaa/gfs/analysis-hourly/latest.zarr": {
        "landing_page": "https://dynamical.org/catalog/noaa-gfs-analysis-hourly/",
    },
    "https://data.dynamical.org/noaa/gefs/analysis/latest.zarr": {
        "landing_page": "https://dynamical.org/catalog/noaa-gefs-analysis",
    },
    "https://data.dynamical.org/noaa/hrrr/analysis/latest.zarr": {
        "landing_page": "https://dynamical.org/catalog/noaa-hrrr-analysis",
    },
}

## Parse metadata


In [ ]:
# Dynamical.org metadata models


class DatasetAttrs(BaseModel):
    dataset_id: str = Field(validation_alias=AliasChoices("dataset_id", "id"))
    dataset_version: str | None = None
    name: str
    description: str
    attribution: str
    spatial_domain: str
    spatial_resolution: str
    time_domain: str
    time_resolution: str
    forecast_domain: str | None = None
    forecast_resolution: str | None = None


class DataVarAttrs(BaseModel):
    long_name: str
    short_name: str | None = None
    standard_name: str | None = None
    units: str
    comment: str | None = None
    step_type: Literal["instant", "accum", "avg", "min", "max"] | None = None
    ensemble_statistic: str | None = None

In [ ]:
class Dimension(BaseModel):
    id: str
    standard_name: str
    field_name: str
    alternate_name: str | None = None
    long_name: str | None = None
    units: str | None = None


class CanonicalDimension(BaseModel):
    standard_name: str
    alternate_names: set[str]

    @classmethod
    def from_dimensions(cls, dims: list[Dimension]) -> "CanonicalDimension":
        assert len({dim.standard_name for dim in dims}) == 1
        alternate_names = {dim.alternate_name for dim in dims if dim.alternate_name is not None}
        field_names = {dim.field_name for dim in dims}
        standard_name = dims[0].standard_name
        return cls(
            standard_name=standard_name,
            alternate_names=alternate_names | field_names - {standard_name},
        )


class Variable(BaseModel):
    id: str
    standard_name: str | None = None
    field_name: str
    long_name: str
    units: str | None = None
    dims: list[Dimension]


class Dataset(BaseModel):
    id: str
    title: str | None = None
    summary: str | None = None
    uri: str
    temporal_resolution: timedelta | None
    start_date: datetime
    end_date: datetime
    variables: list[Variable]
    keywords: list[str]
    dims: list[Dimension]
    metadata_link: str | None


class HistoricalDataset(Dataset):
    pass


class ForecastDataset(Dataset):
    lead_times: list[timedelta]

In [ ]:
class HistoricalDatasetTemporalInfo(BaseModel):
    start_date: datetime
    end_date: datetime | None
    temporal_resolution: timedelta | None


class ForecastDatasetTemporalInfo(BaseModel):
    start_date: datetime
    end_date: datetime | None
    temporal_resolution: timedelta | None
    lead_times: list[timedelta]


class DatasetTemporalInfo(RootModel[HistoricalDatasetTemporalInfo | ForecastDatasetTemporalInfo]):
    pass

In [ ]:
def ds_to_prompt(ds: xr.Dataset) -> str:
    """Convert an xarray Dataset to a string prompt for LLM input."""
    out = dedent(f"""
        Dataset Metadata:
        ```json
        {json.dumps(ds.attrs)}
        ```

        Dataset Dimensions:
        ```json
        {json.dumps(list(ds.dims))}
        ```
    """)
    if "lead_time" in ds:
        out += dedent(f"""
            `lead_time` data:
            ```
            {pretty_repr(ds["lead_time"])}
            ```
        """)
    return out

In [ ]:
def extract_data_from_text[Model: BaseModel](
    *,
    llm: LLM,
    model_type: type[Model],
    system_prompt: str,
    user_prompt: str,
    max_tokens: int = 4000,
) -> Model:
    inference_cfg = LLMInferenceConfig(
        system_prompt=system_prompt,
        json_mode=True,
        max_tokens=max_tokens,
    )
    messages = [LLMMessage(role="user", content=user_prompt)]
    resp = llm.prompt(messages=messages, inference_cfg=inference_cfg)
    out = model_type.model_validate_json(resp.to_text_only())
    return out

In [ ]:
def extract_temporal_info_from_ds(ds: xr.Dataset) -> DatasetTemporalInfo:
    return extract_data_from_text(
        llm=BedrockClaudeLLM(model_id=CLAUDE_4_5_SONNET),
        model_type=DatasetTemporalInfo,
        system_prompt=dedent(f"""
            Extract the temporal information from the given dataset metadata into the output schema below.

            Output Schema:
            ```json
            {DatasetTemporalInfo.model_json_schema()}
            ```
            """),  # noqa: E501
        user_prompt=ds_to_prompt(ds),
    )

In [ ]:
def extract_dataset_metadata(uri: str) -> Dataset:
    ds = xr.open_zarr(uri, chunks="auto")  # pyright: ignore[reportUnknownMemberType]
    ds_metadata = DatasetAttrs.model_validate(ds.attrs)
    vars_metadata: dict[str, DataVarAttrs] = {
        var_name: DataVarAttrs.model_validate(ds[var_name].attrs) for var_name in ds.data_vars
    }  # pyright: ignore[reportAssignmentType]
    dim_names: list[str] = list(ds.dims)  # pyright: ignore[reportAssignmentType]
    dims_out = {
        dim_name: Dimension(
            id=dim_name,
            standard_name=dim_std_names.get(dim_name, dim_name),
            field_name=dim_name,
            alternate_name=dim_alt_names.get(dim_name),
            long_name=ds[dim_name].attrs.get("long_name"),
            units=ds[dim_name].attrs.get("units"),
        )
        for dim_name in dim_names
    }
    vars_out = [
        Variable(
            id=f"{var_name}_{ds_metadata.dataset_id}",
            standard_name=var_metadata.standard_name,
            field_name=var_name,
            long_name=var_metadata.long_name,
            units=var_metadata.units,
            dims=[dims_out[dim_name] for dim_name in ds[var_name].dims],  # pyright: ignore[reportArgumentType]
        )
        for var_name, var_metadata in vars_metadata.items()
    ]

    temporal_info = extract_temporal_info_from_ds(ds)
    if temporal_info.root.end_date is None:
        # use today's date
        temporal_info.root.end_date = datetime.now(tz=UTC)

    if isinstance(temporal_info.root, ForecastDatasetTemporalInfo):
        description_extra_items = ds_metadata.model_dump(
            include={
                "spatial_domain",
                "spatial_resolution",
                "forecast_domain",
                "forecast_resolution",
                "attribution",
            }
        )
        description_extra = "\n".join(f"{k}: {v}" for k, v in description_extra_items.items())
        expanded_description = f"{ds_metadata.description}\n\n{description_extra}"
        out = ForecastDataset(
            id=ds_metadata.dataset_id,
            title=ds_metadata.name,
            summary=expanded_description,
            uri=uri,
            temporal_resolution=temporal_info.root.temporal_resolution,
            start_date=temporal_info.root.start_date.replace(tzinfo=UTC),
            end_date=temporal_info.root.end_date.replace(tzinfo=UTC),
            lead_times=temporal_info.root.lead_times,
            variables=vars_out,
            keywords=[],
            dims=list(dims_out.values()),
            metadata_link=overrides.get(uri, {}).get("landing_page"),
        )
    else:
        description_extra_items = ds_metadata.model_dump(
            include={
                "spatial_domain",
                "spatial_resolution",
                "attribution",
            }
        )
        description_extra = "\n".join(f"{k}: {v}" for k, v in description_extra_items.items())
        expanded_description = f"{ds_metadata.description}\n\n{description_extra}"
        out = HistoricalDataset(
            id=ds_metadata.dataset_id,
            title=ds_metadata.name,
            summary=expanded_description,
            uri=uri,
            temporal_resolution=temporal_info.root.temporal_resolution,
            start_date=temporal_info.root.start_date.replace(tzinfo=UTC),
            end_date=temporal_info.root.end_date.replace(tzinfo=UTC),
            variables=vars_out,
            keywords=[],
            dims=list(dims_out.values()),
            metadata_link=overrides.get(uri, {}).get("landing_page"),
        )
    return out

In [ ]:
datasets: dict[str, list[Dataset]] = {
    "historical": [],
    "forecast": [],
}

In [ ]:
with tqdm(uris["historical"]) as bar:
    datasets["historical"] = [extract_dataset_metadata(uri) for uri in bar]

In [ ]:
with tqdm(uris["forecast"]) as bar:
    datasets["forecast"] = [extract_dataset_metadata(uri) for uri in bar]

---


In [ ]:
%%sh
# Download CF names table if not already present
mkdir -p ./data
wget -O ./data/cf_standard_names.xml https://cfconventions.org/Data/cf-standard-names/76/src/cf-standard-name-table.xml

In [ ]:
from defusedxml.ElementTree import parse


def get_cf_standard_names() -> list[str]:
    cf_names_tree = parse("./data/cf_standard_names.xml")
    cf_names_root = cf_names_tree.getroot()
    assert cf_names_root is not None
    # collect `id` attribute from all entries
    cf_standard_names = [entry.get("id") for entry in cf_names_root.findall("entry")]
    return cf_standard_names  # pyright: ignore[reportReturnType]


In [ ]:
cf_standard_names = set(get_cf_standard_names())

## Populate graph


In [ ]:
from rdflib import DCTERMS, RDF, RDFS, BNode, Graph, Literal, Namespace, URIRef
from rdflib.namespace import DCAT, OWL, QB, SDO, TIME, XSD


In [ ]:
# Define Namespaces
BASE = Namespace("http://kmnlp-demo.noaa.gov/")
QUDT = Namespace("http://qudt.org/3.1.1/schema/qudt/")
UNIT = Namespace("http://qudt.org/3.1.1/vocab/unit/")
CF = Namespace("http://mmisw.org/ont/cf/parameter/")

In [ ]:
def add_dataset(g: Graph, dataset: Dataset, *, canonical_dims: bool = False) -> str:  # noqa: C901, PLR0912
    """Add a dataset to the graph."""
    # add dims
    if not canonical_dims:
        for d in dataset.dims:
            add_dim(g, d)

    # add vars
    for v in dataset.variables:
        add_variable(g, v)

    ds_uri = BASE[dataset.id]

    # rdf
    g.add((ds_uri, RDF.type, DCAT.Dataset))
    g.add((ds_uri, RDF.type, SDO.Dataset))
    g.add((ds_uri, RDF.type, QB.DataSet))

    # schema.org properties
    g.add((ds_uri, SDO.version, Literal("2")))
    if dataset.title:
        g.add((ds_uri, SDO.name, Literal(dataset.title)))
    if dataset.summary:
        g.add((ds_uri, SDO.description, Literal(dataset.summary)))

    if dataset.metadata_link:
        g.add((ds_uri, SDO.url, URIRef(dataset.metadata_link)))

    g.add((ds_uri, SDO.contentUrl, URIRef(dataset.uri)))
    g.add((ds_uri, SDO.isAccessibleForFree, Literal(True)))
    # g.add((ds_uri, SDO.identifier, Literal("")))
    for k in dataset.keywords:
        g.add((ds_uri, SDO.keywords, Literal(k)))
    for dim in dataset.dims:
        dim_iri = BASE[dim.standard_name if canonical_dims else dim.id]
        g.add((ds_uri, SDO.variableMeasured, dim_iri))
    for v in dataset.variables:
        g.add((ds_uri, SDO.variableMeasured, BASE[v.id]))

    # dcterms
    if dataset.title:
        g.add((ds_uri, DCTERMS.title, Literal(dataset.title)))
    if dataset.summary:
        g.add((ds_uri, DCTERMS.description, Literal(dataset.summary)))

    # qb
    for v in dataset.variables:
        v_structure_def_bnode = BNode()
        g.add((v_structure_def_bnode, RDF.type, QB.DataStructureDefinition))
        for dim in v.dims:
            qb_dim_bnode = BNode()
            g.add((qb_dim_bnode, RDF.type, QB.ComponentSpecification))
            dim_iri = BASE[dim.standard_name if canonical_dims else dim.id]
            g.add((qb_dim_bnode, QB.dimension, dim_iri))
            g.add((qb_dim_bnode, BASE.field_name, Literal(dim.field_name)))
            g.add((v_structure_def_bnode, QB.component, qb_dim_bnode))
        qb_var_bnode = BNode()
        g.add((qb_var_bnode, QB.measure, BASE[v.id]))
        g.add((v_structure_def_bnode, QB.component, qb_var_bnode))
        g.add((ds_uri, QB.structure, v_structure_def_bnode))

    # rdfs
    if dataset.title:
        g.add((ds_uri, RDFS.label, Literal(dataset.title)))
    if dataset.summary:
        g.add((ds_uri, RDFS.comment, Literal(dataset.summary)))

    # dcat
    dcat_dist_bnode = BNode()
    g.add((dcat_dist_bnode, RDF.type, DCAT.Distribution))
    g.add((dcat_dist_bnode, DCAT.accessURL, URIRef(dataset.uri)))
    g.add((dcat_dist_bnode, DCAT.mediaType, Literal("application/zarr")))
    g.add((ds_uri, DCAT.distribution, dcat_dist_bnode))
    if dataset.temporal_resolution:
        g.add(
            (
                ds_uri,
                DCAT.temporalResolution,
                Literal(dataset.temporal_resolution, datatype=XSD.duration),
            )
        )
    g.add((ds_uri, DCAT.startDate, Literal(dataset.start_date, datatype=XSD.dateTime)))
    g.add((ds_uri, DCAT.endDate, Literal(dataset.end_date, datatype=XSD.dateTime)))

    return ds_uri


def add_variable(g: Graph, variable: Variable) -> str:
    """Add a variable to the graph."""
    var_uri = BASE[variable.id]

    # rdf
    g.add((var_uri, RDF.type, SDO.PropertyValue))
    g.add((var_uri, RDF.type, QB.MeasureProperty))

    # schema.org properties
    g.add((var_uri, SDO.name, Literal(variable.standard_name)))
    g.add((var_uri, SDO.description, Literal(variable.long_name)))
    g.add((var_uri, SDO.alternateName, Literal(variable.field_name)))
    if variable.units:
        g.add((var_uri, SDO.unitText, Literal(variable.units)))
    g.add((var_uri, SDO.propertyID, QB.measure))

    # rdfs
    g.add((var_uri, RDFS.label, Literal(variable.long_name)))

    # qudt
    # g.add((var_uri, QUDT.hasUnit, UNIT[""]))

    # cf
    if variable.standard_name and variable.standard_name in cf_standard_names:
        g.add((var_uri, BASE.cf_standard_name, CF[variable.standard_name]))

    g.add((var_uri, BASE.field_name, Literal(variable.field_name)))

    return var_uri


def add_dim(g: Graph, dim: Dimension) -> str:
    """Add a dimension to the graph."""
    std_name = dim.standard_name
    dim_uri = BASE[dim.id]

    # rdf
    g.add((dim_uri, RDF.type, SDO.PropertyValue))
    g.add((dim_uri, RDF.type, QB.DimensionProperty))

    # schema.org
    if std_name in dim_alt_names:
        g.add((dim_uri, SDO.alternateName, Literal(dim_alt_names[std_name])))
    if dim.units:
        g.add((dim_uri, SDO.unitText, Literal(dim.units)))
    g.add((dim_uri, SDO.propertyID, QB.measureDimension))

    # rdfs
    g.add((dim_uri, RDFS.label, Literal(dim.long_name)))

    # cf
    if std_name in cf_standard_names:
        g.add((dim_uri, BASE.cf_standard_name, CF[std_name]))

    return dim_uri


def add_dim_canonical(g: Graph, dim: CanonicalDimension) -> str:
    """Add a canonical dimension to the graph."""
    std_name = dim.standard_name
    dim_uri = BASE[std_name]

    # rdf
    g.add((dim_uri, RDF.type, SDO.PropertyValue))
    g.add((dim_uri, RDF.type, QB.DimensionProperty))

    # schema.org
    g.add((dim_uri, SDO.name, Literal(std_name)))
    for alt_name in dim.alternate_names:
        g.add((dim_uri, SDO.alternateName, Literal(alt_name)))
    g.add((dim_uri, SDO.propertyID, QB.measureDimension))

    # rdfs
    g.add((dim_uri, RDFS.label, Literal(std_name)))

    # cf
    if std_name in cf_standard_names:
        g.add((dim_uri, BASE.cf_standard_name, CF[std_name]))

    return dim_uri

In [ ]:
g = Graph()

# bind prefixes
g.bind("", BASE)
g.bind("cf", CF)
g.bind("dcat", DCAT)
g.bind("dcterms", DCTERMS)
g.bind("owl", OWL)
g.bind("qb", QB)
g.bind("qudt", QUDT)
g.bind("rdf", RDF)
g.bind("rdfs", RDFS)
g.bind("schema", SDO)
g.bind("time", TIME)
g.bind("unit", UNIT)
g.bind("xsd", XSD)

# custom properties
cf_standard_name_prop = BASE["cf_standard_name"]
g.add((cf_standard_name_prop, RDF.type, RDF.Property))
g.add(
    (cf_standard_name_prop, RDFS.label, Literal("Matching entry in the CF Standard Names table."))
)
g.add((cf_standard_name_prop, RDFS.range, CF.Standard_Name))

field_name_prop = BASE["field_name"]
g.add((field_name_prop, RDF.type, RDF.Property))
g.add((field_name_prop, RDFS.label, Literal("Field name in the dataset.")))
g.add((field_name_prop, RDFS.range, XSD.string))

if USE_CANONICAL_DIMS:
    std_name_to_dims: dict[str, list[Dimension]] = {}
    for ds in datasets["historical"] + datasets["forecast"]:
        for dim in ds.dims:
            if dim.standard_name not in std_name_to_dims:
                std_name_to_dims[dim.standard_name] = []
            std_name_to_dims[dim.standard_name].append(dim)

    for dims in std_name_to_dims.values():
        add_dim_canonical(g, CanonicalDimension.from_dimensions(dims))

for ds in datasets["historical"] + datasets["forecast"]:
    add_dataset(g, ds, canonical_dims=USE_CANONICAL_DIMS)

## Save


In [ ]:
ttl_str = g.serialize(format="turtle")

In [ ]:
out_path = "../../src/llm_agent/data/dataset-kg-dynamical-org-generated.ttl"
with open(out_path, "w") as f:
    f.write(ttl_str.rstrip())
    f.write("\n")